# HDFS_v1 Log Anomaly Detection with Deep Learning (DeepLog-style LSTM)

This Kaggle notebook implements a complete prototype for your project:

**HDFS logs → preprocessing/parsing → tokenization/log templates → sequence creation → LSTM model → anomaly/normal prediction → precision, recall, F1-score**

It is designed for the **HDFS_v1** dataset, usually containing:

- `HDFS.log`
- `anomaly_label.csv`

Before running, add the HDFS_v1 dataset to your Kaggle notebook using **Add Input**.

> Project context: intrusion/anomaly detection from HDFS logs using Deep Learning and NLP-style sequence modeling.

## 1. Install / import libraries

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Find the HDFS_v1 files in Kaggle input

In [ ]:
import os

def find_file(filename, root="/kaggle/input"):
    matches = []
    for dirname, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower() == filename.lower():
                matches.append(os.path.join(dirname, f))
    return matches

print("Files available in /kaggle/input:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))

hdfs_log_candidates = find_file("HDFS.log")
label_candidates = find_file("anomaly_label.csv")

print()
print("HDFS.log candidates:", hdfs_log_candidates)
print("anomaly_label.csv candidates:", label_candidates)

if not hdfs_log_candidates:
    raise FileNotFoundError("HDFS.log not found. Add the HDFS_v1 dataset as Kaggle input.")
if not label_candidates:
    raise FileNotFoundError("anomaly_label.csv not found. Add the HDFS_v1 dataset as Kaggle input.")

HDFS_LOG_PATH = hdfs_log_candidates[0]
LABEL_PATH = label_candidates[0]

print()
print("Using:")
print("HDFS_LOG_PATH =", HDFS_LOG_PATH)
print("LABEL_PATH    =", LABEL_PATH)

## 3. Load labels

The HDFS_v1 labels are usually given by **BlockId** and **Label**. Labels are often `Normal` or `Anomaly`.

In [ ]:
labels_df = pd.read_csv(LABEL_PATH)

print('Labels head:')
print(labels_df.head())
print('Columns:', list(labels_df.columns))

# Normalize column names if needed
if 'BlockId' not in labels_df.columns:
    possible_cols = [c for c in labels_df.columns if 'block' in c.lower()]
    if possible_cols:
        labels_df = labels_df.rename(columns={possible_cols[0]: 'BlockId'})

if 'Label' not in labels_df.columns:
    possible_cols = [c for c in labels_df.columns if 'label' in c.lower()]
    if possible_cols:
        labels_df = labels_df.rename(columns={possible_cols[0]: 'Label'})

if 'BlockId' not in labels_df.columns or 'Label' not in labels_df.columns:
    raise ValueError('Could not find BlockId and Label columns in anomaly_label.csv')

labels_df['y'] = labels_df['Label'].apply(lambda x: 0 if str(x).lower() == 'normal' else 1)
label_map = dict(zip(labels_df['BlockId'], labels_df['y']))

print()
print('Encoded labels: 0 = Normal, 1 = Anomaly')
print(labels_df[['BlockId', 'Label', 'y']].head())
print()
print('Label distribution:')
print(labels_df['y'].value_counts())

## 4. Parse raw HDFS logs

Each HDFS log line contains a block ID like `blk_...`. We group log events by block ID, because each block becomes one sequence.

In [ ]:
import re
# Correct block pattern: detects blk_123 and blk_-123
block_pattern = re.compile(r'blk_-?\d+')

# More flexible HDFS log parser
line_pattern = re.compile(
    r'(?P<Date>\d{6})\s+'
    r'(?P<Time>\d{6})\s+'
    r'(?P<Pid>\d+)\s+'
    r'(?P<Level>\w+)\s+'
    r'(?P<Component>[^\s:]+):\s+'
    r'(?P<Content>.*)'
)

def extract_block_id(line):
    match = block_pattern.search(line)
    return match.group(0) if match else None

def parse_line(line):
    m = line_pattern.match(line)
    if m:
        d = m.groupdict()
        return d['Date'], d['Time'], d['Pid'], d['Level'], d['Component'], d['Content']
    return None, None, None, None, None, line.strip()

records = []

with open(HDFS_LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        block_id = extract_block_id(line)

        if block_id is None:
            continue

        date, time, pid, level, component, content = parse_line(line)

        records.append((
            i,
            block_id,
            date,
            time,
            pid,
            level,
            component,
            content,
            line.strip()
        ))

logs_df = pd.DataFrame(
    records,
    columns=['LineId', 'BlockId', 'Date', 'Time', 'Pid', 'Level', 'Component', 'Content', 'Raw']
)

logs_df = logs_df[logs_df['BlockId'].isin(label_map)].copy()
logs_df['y'] = logs_df['BlockId'].map(label_map)

print('Parsed log events:', len(logs_df))
print('Unique blocks:', logs_df['BlockId'].nunique())
print()
print(logs_df.head())
print()
print('Event labels distribution:')
print(logs_df['y'].value_counts())

## 5. Create simple log templates / tokens

In real log analysis, tools like **Drain** are often used to extract log templates.

Here we use a simple regex-based template function so the notebook is ready to run without external tools:

- replace block IDs with `<BLK>`
- replace IP addresses with `<IP>`
- replace numbers with `<NUM>`
- keep the general message structure

Each unique template becomes a token ID.

In [ ]:
def make_template(content):
    text = str(content)
    text = re.sub(r'blk_-?\d+', '<BLK>', text)
    text = re.sub(r'\b\d+\.\d+\.\d+\.\d+\b', '<IP>', text)
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    text = re.sub(r'(/[^\s]+)+', '<PATH>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

logs_df['Template'] = logs_df['Content'].apply(make_template)

# Map templates to integer token ids. 0 is reserved for padding.
template_counts = Counter(logs_df['Template'])
templates = [tpl for tpl, _ in template_counts.most_common()]
template_to_id = {tpl: i + 1 for i, tpl in enumerate(templates)}
id_to_template = {i: tpl for tpl, i in template_to_id.items()}

logs_df['EventId'] = logs_df['Template'].map(template_to_id)

print('Number of unique templates:', len(template_to_id))
print('\nMost common templates:')
for tpl, count in template_counts.most_common(10):
    print(count, '=>', tpl)

logs_df[['BlockId', 'Content', 'Template', 'EventId', 'y']].head()

## 6. Build one sequence per HDFS block

For each `BlockId`, we create an ordered sequence of template IDs.

The label is:

- `0`: normal
- `1`: anomaly / intrusion-like behavior

In [ ]:
# Sort by original line order to preserve sequence order
logs_df = logs_df.sort_values(['BlockId', 'LineId'])

# Create one EventId sequence per HDFS block
sequences = logs_df.groupby('BlockId')['EventId'].apply(list).reset_index(name='Sequence')

# Add label for each block
sequences['y'] = sequences['BlockId'].map(label_map)

# Add sequence length
sequences['Length'] = sequences['Sequence'].apply(len)

print(sequences.head())

print()
print('Number of sequences:', len(sequences))

print()
print('Block label distribution:')
print(sequences['y'].value_counts())

print()
print('Sequence length statistics:')
print(sequences['Length'].describe())

## 7. Pad / truncate sequences

Neural networks need fixed-size batches. We keep at most `MAX_LEN` events per block. Shorter sequences are padded with `0`.

In [ ]:
MAX_LEN = int(min(100, max(10, sequences['Length'].quantile(0.95))))
print('MAX_LEN =', MAX_LEN)

def pad_sequence(seq, max_len=MAX_LEN):
    seq = seq[:max_len]
    if len(seq) < max_len:
        seq = seq + [0] * (max_len - len(seq))
    return seq

X = np.array([pad_sequence(seq) for seq in sequences['Sequence']], dtype=np.int64)
y = sequences['y'].values.astype(np.int64)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Vocabulary size including PAD:', len(template_to_id) + 1)

## 8. Train / validation / test split

We use stratified splitting to keep both normal and anomaly examples in each set.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Train:', X_train.shape, Counter(y_train))
print('Val:  ', X_val.shape, Counter(y_val))
print('Test: ', X_test.shape, Counter(y_test))

## 9. PyTorch dataset and dataloaders

In [ ]:
class HDFSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 128

train_loader = DataLoader(HDFSDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(HDFSDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(HDFSDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 10. DeepLog-style LSTM model

This model treats each log block as a sequence of log-template tokens.

Architecture:

`Template IDs → Embedding → LSTM → Dense layer → Normal/Anomaly prediction`

In [ ]:
class LogLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (h_n, c_n) = self.lstm(emb)
        last_hidden = h_n[-1]
        logits = self.fc(self.dropout(last_hidden))
        return logits

vocab_size = len(template_to_id) + 1
model = LogLSTMClassifier(vocab_size=vocab_size).to(device)
print(model)

## 11. Training functions

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            labels = yb.detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, acc, f1

# Handle class imbalance by weighting anomaly class
class_counts = Counter(y_train)
weights = torch.tensor([
    len(y_train) / (2 * class_counts.get(0, 1)),
    len(y_train) / (2 * class_counts.get(1, 1))
], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('Class weights:', weights.detach().cpu().numpy())

## 12. Train the model

In [ ]:
EPOCHS = 5
best_val_f1 = -1
best_path = 'best_hdfs_lstm_model.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion, optimizer=None)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_path)

    print(
        f'Epoch {epoch:02d} | '
        f'Train loss {train_loss:.4f}, acc {train_acc:.4f}, f1 {train_f1:.4f} | '
        f'Val loss {val_loss:.4f}, acc {val_acc:.4f}, f1 {val_f1:.4f}'
    )

print('Best validation F1:', best_val_f1)

## 13. Evaluation on the Test Set

In this section, the best saved LSTM model is evaluated on the test set. The results are presented using:

- **Accuracy**: overall percentage of correct predictions.
- **Precision**: among predicted anomalies, how many are truly anomalies.
- **Recall**: among real anomalies, how many were detected.
- **F1-score**: harmonic mean between precision and recall.
- **Confusion matrix**: visual comparison between true labels and predicted labels.

For anomaly detection, **recall and F1-score are especially important**, because the objective is not only to obtain high accuracy, but also to correctly detect anomalous sequences.


In [ ]:
# 13. Evaluate on the test set with professional tables and figures

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

# Load the best saved model
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)

        # Probability of class 1 = Anomaly
        probs = torch.softmax(logits, dim=1)[:, 1]

        # Predicted class: 0 = Normal, 1 = Anomaly
        preds = logits.argmax(dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

# Convert lists to numpy arrays
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ==============================
# 1) Main evaluation metrics
# ==============================
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1],
    "Score (%)": [accuracy * 100, precision * 100, recall * 100, f1 * 100]
})

metrics_df["Score"] = metrics_df["Score"].round(4)
metrics_df["Score (%)"] = metrics_df["Score (%)"].round(2)

print("Main evaluation metrics on the test set:")
display(metrics_df)

# Save metrics for the report
metrics_df.to_csv("test_metrics.csv", index=False)

# ==============================
# 2) Classification report table
# ==============================
report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=["Normal", "Anomaly"],
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.round(4)

print("Detailed classification report:")
display(report_df)

# Save detailed report
report_df.to_csv("classification_report.csv")

# ==============================
# 3) Confusion matrix as table
# ==============================
cm = confusion_matrix(all_labels, all_preds)

cm_df = pd.DataFrame(
    cm,
    index=["True Normal", "True Anomaly"],
    columns=["Predicted Normal", "Predicted Anomaly"]
)

print("Confusion matrix table:")
display(cm_df)

cm_df.to_csv("confusion_matrix.csv")

# ==============================
# 4) Confusion matrix figure
# ==============================
fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Anomaly"]
)

disp.plot(cmap="Blues", ax=ax, colorbar=False, values_format="d")

ax.set_title("Confusion Matrix on the Test Set", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
plt.grid(False)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# ==============================
# 5) Bar chart of metrics
# ==============================
plt.figure(figsize=(7, 5))
metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
metric_values = [accuracy, precision, recall, f1]

bars = plt.bar(metric_names, metric_values)
plt.ylim(0, 1.05)
plt.title("Evaluation Metrics on the Test Set", fontsize=14, fontweight="bold")
plt.ylabel("Score", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)

for bar, value in zip(bars, metric_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.tight_layout()
plt.savefig("evaluation_metrics_bar_chart.png", dpi=300, bbox_inches="tight")
plt.show()

# ==============================
# 6) ROC curve and AUC score
# ==============================
# ROC is useful when the model outputs probabilities.
# It shows the trade-off between true positive rate and false positive rate.
try:
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
    plt.title("ROC Curve on the Test Set", fontsize=14, fontweight="bold")
    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.legend(loc="lower right")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("roc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    print(f"AUC score: {roc_auc:.4f}")

except ValueError:
    print("ROC curve could not be computed because only one class is present in the test labels.")

print("Saved evaluation files:")
print("- test_metrics.csv")
print("- classification_report.csv")
print("- confusion_matrix.csv")
print("- confusion_matrix.png")
print("- evaluation_metrics_bar_chart.png")
print("- roc_curve.png, if ROC could be computed")


### How to interpret these results

The metrics table gives a compact summary of the model performance. Accuracy measures the global correctness of the model, while precision and recall focus on anomaly detection quality. A high precision means that most predicted anomalies are correct. A high recall means that the model detects most real anomalies. The F1-score balances both precision and recall, making it a useful metric when the dataset is imbalanced.

The confusion matrix gives a clearer view of the errors. The diagonal values represent correct predictions, while the off-diagonal values represent misclassifications. In this project, false negatives are important because they correspond to anomalous log sequences classified as normal.


## 14. Show example predictions

This helps you explain the model output in your presentation.

In [ ]:
test_indices = np.arange(len(X_test))
examples = pd.DataFrame({
    'true_label': all_labels,
    'pred_label': all_preds,
    'anomaly_probability': all_probs
})
examples['true_label_text'] = examples['true_label'].map({0: 'Normal', 1: 'Anomaly'})
examples['pred_label_text'] = examples['pred_label'].map({0: 'Normal', 1: 'Anomaly'})
examples.head(10)

## 15. Explainable AI (XAI): explain why the LSTM predicts an anomaly

In this notebook, the model does not read raw text directly. It reads a sequence of **log-template IDs**.  
So, the most useful explanation is:

> Which log events/templates contributed the most to the final prediction?

We add two XAI methods:

1. **Gradient × Embedding saliency**: shows which tokens influenced the neural network internally.
2. **Occlusion / perturbation importance**: removes one event at a time and checks how much the anomaly probability changes.

These methods help explain predictions in an academic report or presentation.


In [ ]:
import matplotlib.pyplot as plt

# Make sure the best model is loaded
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

def decode_sequence(seq):
    """
    Convert token IDs back to readable log templates.
    Padding token 0 is ignored.
    """
    decoded = []
    for token_id in seq:
        token_id = int(token_id)
        if token_id != 0:
            decoded.append(id_to_template.get(token_id, f"UNKNOWN_TOKEN_{token_id}"))
    return decoded


def predict_one_sequence(seq):
    """
    Return predicted class and anomaly probability for one padded sequence.
    """
    model.eval()
    x = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(torch.argmax(probs).cpu().item())
        anomaly_prob = float(probs[1].cpu().item())

    return pred, anomaly_prob


### 15.1 Local explanation using Gradient × Embedding

This method explains **one prediction**.  
It calculates how sensitive the model output is to each event in the sequence.

A high score means the event had a stronger influence on the prediction.


In [ ]:
def gradient_x_embedding_explanation(seq, target_class=1):
    """
    Explain one sequence using Gradient x Embedding.

    target_class = 1 explains the anomaly score.
    target_class = 0 explains the normal score.
    """

    # Save current model mode
    was_training = model.training

    # cuDNN LSTM needs training mode for backward
    model.train()

    # But we keep dropout disabled for stable explanation
    model.dropout.eval()

    x = torch.tensor(seq, dtype=torch.long).unsqueeze(0).to(device)

    # Clear previous gradients
    model.zero_grad(set_to_none=True)

    # Get embeddings and keep gradients
    emb = model.embedding(x)
    emb.retain_grad()

    # Forward pass manually from embeddings
    out, (h_n, c_n) = model.lstm(emb)
    last_hidden = h_n[-1]

    # Do NOT apply dropout for XAI stability
    logits = model.fc(last_hidden)

    # Backpropagate selected class score
    score = logits[0, target_class]
    score.backward()

    # Gradient x Embedding importance
    importance = (emb.grad * emb).sum(dim=2).abs().squeeze(0).detach().cpu().numpy()

    # Ignore PAD tokens
    seq_np = np.array(seq)
    importance[seq_np == 0] = 0

    # Restore previous model mode
    if was_training:
        model.train()
    else:
        model.eval()

    return importance

def explain_test_example_gradient(example_index, target_class=1, top_k=10):
    seq = X_test[example_index]
    true_label = int(y_test[example_index])

    pred_label, anomaly_prob = predict_one_sequence(seq)
    importance = gradient_x_embedding_explanation(seq, target_class=target_class)

    rows = []
    for position, token_id in enumerate(seq):
        token_id = int(token_id)
        if token_id == 0:
            continue

        rows.append({
            "position": position,
            "event_id": token_id,
            "template": id_to_template.get(token_id, "UNKNOWN"),
            "importance": float(importance[position])
        })

    explanation_df = pd.DataFrame(rows).sort_values("importance", ascending=False)

    print("Example index:", example_index)
    print("True label:", "Anomaly" if true_label == 1 else "Normal")
    print("Predicted label:", "Anomaly" if pred_label == 1 else "Normal")
    print("Anomaly probability:", round(anomaly_prob, 4))

    display(explanation_df.head(top_k))

    # Plot top important templates
    plot_df = explanation_df.head(top_k).copy()
    plot_df["short_template"] = plot_df["template"].str.slice(0, 80)

    plt.figure(figsize=(10, 5))
    plt.barh(plot_df["short_template"], plot_df["importance"])
    plt.gca().invert_yaxis()
    plt.xlabel("Importance score")
    plt.ylabel("Log template")
    plt.title("XAI: Most influential log templates using Gradient × Embedding")
    plt.show()

    return explanation_df


# Choose an anomalous example if possible, otherwise use the first test example
anomaly_positions = np.where(y_test == 1)[0]
example_index = int(anomaly_positions[0]) if len(anomaly_positions) > 0 else 0

gradient_explanation_df = explain_test_example_gradient(
    example_index=example_index,
    target_class=1,
    top_k=10
)


### 15.2 Local explanation using Occlusion / Perturbation

This method is easier to explain in a report:

> We remove one log event at a time by replacing it with padding.  
> If the anomaly probability decreases strongly, that event was important for the anomaly decision.

This is model-agnostic because it only compares model outputs before and after perturbation.


In [ ]:
def occlusion_explanation(seq):
    """
    Explain one sequence by replacing each event with PAD=0
    and measuring the change in anomaly probability.
    """
    base_pred, base_prob = predict_one_sequence(seq)

    rows = []
    for position, token_id in enumerate(seq):
        token_id = int(token_id)
        if token_id == 0:
            continue

        perturbed_seq = seq.copy()
        perturbed_seq[position] = 0

        _, new_prob = predict_one_sequence(perturbed_seq)

        rows.append({
            "position": position,
            "event_id": token_id,
            "template": id_to_template.get(token_id, "UNKNOWN"),
            "base_anomaly_probability": base_prob,
            "probability_after_removal": new_prob,
            "importance_drop": base_prob - new_prob
        })

    explanation_df = pd.DataFrame(rows).sort_values("importance_drop", ascending=False)
    return explanation_df, base_pred, base_prob


def explain_test_example_occlusion(example_index, top_k=10):
    seq = X_test[example_index]
    true_label = int(y_test[example_index])

    explanation_df, pred_label, base_prob = occlusion_explanation(seq)

    print("Example index:", example_index)
    print("True label:", "Anomaly" if true_label == 1 else "Normal")
    print("Predicted label:", "Anomaly" if pred_label == 1 else "Normal")
    print("Original anomaly probability:", round(base_prob, 4))

    display(explanation_df.head(top_k))

    plot_df = explanation_df.head(top_k).copy()
    plot_df["short_template"] = plot_df["template"].str.slice(0, 80)

    plt.figure(figsize=(10, 5))
    plt.barh(plot_df["short_template"], plot_df["importance_drop"])
    plt.gca().invert_yaxis()
    plt.xlabel("Drop in anomaly probability after removing the event")
    plt.ylabel("Log template")
    plt.title("XAI: Most influential log templates using Occlusion")
    plt.show()

    return explanation_df


occlusion_explanation_df = explain_test_example_occlusion(
    example_index=example_index,
    top_k=10
)


### 15.3 Global XAI: most important templates over many test samples

Local XAI explains one prediction.  
Global XAI gives a general view of what the model often uses to detect anomalies.

Here, we calculate Gradient × Embedding importance for several test examples and aggregate the results by template.


In [ ]:
def global_template_importance(X_data, y_data=None, max_samples=200, target_class=1):
    """
    Aggregate token importance over many sequences.
    """
    rows = []

    n = min(max_samples, len(X_data))

    for i in range(n):
        seq = X_data[i]
        importance = gradient_x_embedding_explanation(seq, target_class=target_class)

        for position, token_id in enumerate(seq):
            token_id = int(token_id)
            if token_id == 0:
                continue

            rows.append({
                "event_id": token_id,
                "template": id_to_template.get(token_id, "UNKNOWN"),
                "importance": float(importance[position]),
                "label": int(y_data[i]) if y_data is not None else None
            })

    importance_df = pd.DataFrame(rows)

    global_df = (
        importance_df
        .groupby(["event_id", "template"], as_index=False)
        .agg(
            mean_importance=("importance", "mean"),
            total_importance=("importance", "sum"),
            frequency=("importance", "count")
        )
        .sort_values("mean_importance", ascending=False)
    )

    return global_df


global_xai_df = global_template_importance(
    X_test,
    y_test,
    max_samples=200,
    target_class=1
)

display(global_xai_df.head(15))

plot_df = global_xai_df.head(10).copy()
plot_df["short_template"] = plot_df["template"].str.slice(0, 80)

plt.figure(figsize=(10, 5))
plt.barh(plot_df["short_template"], plot_df["mean_importance"])
plt.gca().invert_yaxis()
plt.xlabel("Mean importance")
plt.ylabel("Log template")
plt.title("Global XAI: most important log templates for anomaly detection")
plt.show()


### 15.4 Save XAI results


In [ ]:
gradient_explanation_df.to_csv("xai_gradient_example.csv", index=False)
occlusion_explanation_df.to_csv("xai_occlusion_example.csv", index=False)
global_xai_df.to_csv("xai_global_template_importance.csv", index=False)

print("Saved XAI files:")
print("- xai_gradient_example.csv")
print("- xai_occlusion_example.csv")
print("- xai_global_template_importance.csv")


## 16. Save outputs

In [ ]:
import json

with open('template_to_id.json', 'w') as f:
    json.dump(template_to_id, f, indent=2)

examples.to_csv('hdfs_lstm_test_predictions.csv', index=False)

print('Saved files:')
print('- best_hdfs_lstm_model.pt')
print('- template_to_id.json')
print('- hdfs_lstm_test_predictions.csv')
print('- test_metrics.csv')
print('- classification_report.csv')
print('- confusion_matrix.csv')
print('- confusion_matrix.png')
print('- evaluation_metrics_bar_chart.png')
print('- roc_curve.png, if ROC could be computed')
